In [1]:
# Compare ways of creating coordinate sequence

In [2]:
import torch

In [3]:
win_size = (5, 5)
win_h, win_w = win_size

In [4]:
# Official approach
coords_h = torch.arange(win_h)
coords_w = torch.arange(win_w)
coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
print(relative_coords)
print(relative_coords.shape)

tensor([[[ 0,  0],
         [ 0, -1],
         [ 0, -2],
         ...,
         [-4, -2],
         [-4, -3],
         [-4, -4]],

        [[ 0,  1],
         [ 0,  0],
         [ 0, -1],
         ...,
         [-4, -1],
         [-4, -2],
         [-4, -3]],

        [[ 0,  2],
         [ 0,  1],
         [ 0,  0],
         ...,
         [-4,  0],
         [-4, -1],
         [-4, -2]],

        ...,

        [[ 4,  2],
         [ 4,  1],
         [ 4,  0],
         ...,
         [ 0,  0],
         [ 0, -1],
         [ 0, -2]],

        [[ 4,  3],
         [ 4,  2],
         [ 4,  1],
         ...,
         [ 0,  1],
         [ 0,  0],
         [ 0, -1]],

        [[ 4,  4],
         [ 4,  3],
         [ 4,  2],
         ...,
         [ 0,  2],
         [ 0,  1],
         [ 0,  0]]])
torch.Size([25, 25, 2])


/Users/ian/projects/mmdummy/.envs/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647065806/work/aten/src/ATen/native/TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [5]:
# My approach, Part 1
coord_h = torch.arange(win_h)
coord_w = torch.arange(win_w)
coords_flatten = torch.cartesian_prod(coord_h, coord_w)
print(coords_flatten)
print(coords_flatten.shape)

tensor([[0, 0],
        [0, 1],
        [0, 2],
        [0, 3],
        [0, 4],
        [1, 0],
        [1, 1],
        [1, 2],
        [1, 3],
        [1, 4],
        [2, 0],
        [2, 1],
        [2, 2],
        [2, 3],
        [2, 4],
        [3, 0],
        [3, 1],
        [3, 2],
        [3, 3],
        [3, 4],
        [4, 0],
        [4, 1],
        [4, 2],
        [4, 3],
        [4, 4]])
torch.Size([25, 2])


In [6]:
# My approach, Part 2
my_relative_coords = coords_flatten[:, None, :] - coords_flatten[None, :, :]
print(my_relative_coords)
print(my_relative_coords.shape)

tensor([[[ 0,  0],
         [ 0, -1],
         [ 0, -2],
         ...,
         [-4, -2],
         [-4, -3],
         [-4, -4]],

        [[ 0,  1],
         [ 0,  0],
         [ 0, -1],
         ...,
         [-4, -1],
         [-4, -2],
         [-4, -3]],

        [[ 0,  2],
         [ 0,  1],
         [ 0,  0],
         ...,
         [-4,  0],
         [-4, -1],
         [-4, -2]],

        ...,

        [[ 4,  2],
         [ 4,  1],
         [ 4,  0],
         ...,
         [ 0,  0],
         [ 0, -1],
         [ 0, -2]],

        [[ 4,  3],
         [ 4,  2],
         [ 4,  1],
         ...,
         [ 0,  1],
         [ 0,  0],
         [ 0, -1]],

        [[ 4,  4],
         [ 4,  3],
         [ 4,  2],
         ...,
         [ 0,  2],
         [ 0,  1],
         [ 0,  0]]])
torch.Size([25, 25, 2])


In [7]:
torch.equal(relative_coords, my_relative_coords)

True

In [8]:
from mmpose.models.backbones.rsn_swin import SwinStep

/Users/ian/projects/mmdummy/.envs/lib/python3.10/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [9]:
# My approach, Class static method
rel_pos_to_bias_bucket = SwinStep.compute_rel_pos_to_bias_bucket(win_h, win_w)
print(rel_pos_to_bias_bucket)
print(rel_pos_to_bias_bucket.shape)

tensor([[40, 39, 38, 37, 36, 31, 30, 29, 28, 27, 22, 21, 20, 19, 18, 13, 12, 11,
         10,  9,  4,  3,  2,  1,  0],
        [41, 40, 39, 38, 37, 32, 31, 30, 29, 28, 23, 22, 21, 20, 19, 14, 13, 12,
         11, 10,  5,  4,  3,  2,  1],
        [42, 41, 40, 39, 38, 33, 32, 31, 30, 29, 24, 23, 22, 21, 20, 15, 14, 13,
         12, 11,  6,  5,  4,  3,  2],
        [43, 42, 41, 40, 39, 34, 33, 32, 31, 30, 25, 24, 23, 22, 21, 16, 15, 14,
         13, 12,  7,  6,  5,  4,  3],
        [44, 43, 42, 41, 40, 35, 34, 33, 32, 31, 26, 25, 24, 23, 22, 17, 16, 15,
         14, 13,  8,  7,  6,  5,  4],
        [49, 48, 47, 46, 45, 40, 39, 38, 37, 36, 31, 30, 29, 28, 27, 22, 21, 20,
         19, 18, 13, 12, 11, 10,  9],
        [50, 49, 48, 47, 46, 41, 40, 39, 38, 37, 32, 31, 30, 29, 28, 23, 22, 21,
         20, 19, 14, 13, 12, 11, 10],
        [51, 50, 49, 48, 47, 42, 41, 40, 39, 38, 33, 32, 31, 30, 29, 24, 23, 22,
         21, 20, 15, 14, 13, 12, 11],
        [52, 51, 50, 49, 48, 43, 42, 41, 40, 39,